# 089 — Control estructural y edición generativa

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Más fuerza = más pasos de denoising y más costo: s no solo decide
cuánto se conserva, también cuánto se computa.

**Ejercicio 2.** El forward inicial es 0 (modelo intacto) pero los gradientes de
los pesos no lo son: la zero-convolution aprende desde el primer paso. Tras una
actualización W = −0.2 y b = −0.08, el forward ya es distinto de cero — el control
se "enciende" progresivamente.

**Ejercicio 3.** (a) **Inpainting**: hay una región delimitada y el resto debe
quedar intacto. (b) **ControlNet** de pose: es la única técnica que impone
estructura espacial densa de una referencia externa. (c) **img2img** con s ≈
0.4-0.6: se quiere conservar la composición global pero cambiar el estilo.


In [ ]:
# Ejercicio 1 — img2img: paso inicial, pasos de denoising y costo con CFG
T = 50
for s in (0.15, 0.4, 0.6, 0.95):
    t_inicio = round(s * T)
    evals_unet = t_inicio * 2  # CFG: pasada condicional + incondicional
    print(f"s={s:>4}: t_inicio={t_inicio:>2}  pasos={t_inicio:>2}  evaluaciones U-Net={evals_unet}")


In [ ]:
# Ejercicio 2 — zero-convolution: gradientes y forward tras actualizar
W, b, x, g, lr = 0.0, 0.0, 2.5, 0.8, 0.1

y0 = W * x + b
dW = g * x      # gradiente respecto al peso: proporcional a la ENTRADA
db = g          # gradiente respecto al sesgo
dx = g * W      # gradiente hacia la entrada: 0 mientras W = 0
print(f"forward inicial y = {y0}")
print(f"dL/dW = {dW}   dL/db = {db}   dL/dx = {dx}")

W = W - lr * dW
b = b - lr * db
y1 = W * x + b
print(f"tras actualizar: W = {W}  b = {b}")
print(f"nuevo forward con x = {x}: y = {y1}")


**Ejercicio 4.** El contrato mínimo se valida sin asumir valores internos:


In [ ]:
result = run_lab("generation", seed=89)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


## Reflexión

1. ¿Por qué inicializar las zero-convolutions en cero protege al modelo congelado en los primeros pasos y, aun así, no impide que el ControlNet aprenda? Distingue gradiente respecto a los pesos y gradiente hacia la entrada.
2. En img2img, ¿por qué la fuerza s conserva "tipo de información" (frecuencias bajas: composición, masas de color) y no un porcentaje de píxeles concretos?
3. Si el prompt dice "persona sentada" pero el mapa de pose del ControlNet muestra una figura de pie, ¿qué esperas que domine y por qué? ¿Qué papel juega el peso (conditioning scale) del ControlNet?
